In [1]:
from scMPRAforge.core import HypothesisSet, ResultSet, HypothesisTester, table_type
import scMPRAforge as scm

2025-08-18 10:55:14.362971: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-18 10:55:14.367366: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
import pandas as pd

In [3]:
# Hypotheses (zero-reference)
hyp = pd.DataFrame({
    "comparison_CRE": ["CRE1","CRE2"],
    "comparison_cell_type": ["K562","HepG2"],
    "reference_CRE": [pd.NA, pd.NA],
    "reference_cell_type": [pd.NA, pd.NA],
    "meta": ["neg_ctrl","emvar"]
})
print("table_type:", table_type(hyp.columns))
hs = HypothesisSet.from_dataframe(hyp)
print("is_zero_reference:", hs.is_zero_reference().tolist())

table_type: hypotheses
is_zero_reference: [True, True]


In [4]:

# Dummy tester
def test_fn(h):
    return dict(test_statistic=2.0, p_value=0.01, fold_change=1.5, flattened=False, test_type="dummy")

runner = HypothesisTester(test_fn, "dummy")
res = runner.run(hs)
display(res.to_dataframe())

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,test_type,bh_p
0,CRE1,K562,<NA>,<NA>,neg_ctrl,2.0,0.01,1.5,False,dummy,0.01
1,CRE2,HepG2,<NA>,<NA>,emvar,2.0,0.01,1.5,False,dummy,0.02


## Test wald test

In [5]:

#create dask cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client = Client(cluster)

path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial"

import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
    shendure=primordial.training_data
else:
    print("[+] Model not found. Creating...")

    #load data
    data_root="/gpfs/gibbs/pi/reilly/tabula_data"
    shendure=scm.scMPRA_data.from_tsv(f"{data_root}/shendure/shendure_counts_grouped.txt")
    
    shendure.set_negative_controls(["minP","noP"])
    shendure.set_reference_cell("Pluripotent")
    shendure.ortho_filter()

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=shendure)
    primordial.extract_params(client)
    primordial.save(path,name)

[+] Model found. Loading...


In [6]:
# Build a small hypothesis table
hyp_df = pd.DataFrame({
    "comparison_CRE": ["CRE1","CRE2","CRE3"],
    "comparison_cell_type": ["NeuroectodermBrain","NeuroectodermBrain","Pluripotent"],
    # leave reference_* NA to mean "vs baseline in that model" (zero / reference level)
    "reference_CRE": [pd.NA, pd.NA, pd.NA],
    "reference_cell_type": [pd.NA, pd.NA, pd.NA],
    "meta": ["emvar","neg_ctrl","misc"]
})
hs = HypothesisSet.from_dataframe(hyp_df)

# Build the row tester and run
test_fn = build_wald_test_fn(primordial, shendure)
runner = HypothesisTester(test_fn=test_fn, test_type_name="wald")
wald_results = runner.run(hs).to_dataframe()
wald_results.head()

NameError: name 'build_wald_test_fn' is not defined